# Assignment 5 — Part 2: Transfer Learning for Text (Sentiment Analysis with a Transformer)

## 1. Install dependencies

In [1]:
!pip -q install transformers datasets scikit-learn accelerate

## 2. Load the data

The CSVs have many columns but we only need `text` and `sentiment`. We drop rows where either is missing.

In [21]:
import pandas as pd

train_df = pd.read_csv('sentiment_train.csv', encoding_errors='ignore')[['text', 'sentiment']]
test_df  = pd.read_csv('sentiment_test.csv',  encoding_errors='ignore')[['text', 'sentiment']]

train_df = train_df.dropna(subset=['text', 'sentiment']).reset_index(drop=True)
test_df  = test_df.dropna(subset=['text', 'sentiment']).reset_index(drop=True)

print('train shape:', train_df.shape)
print('test  shape:', test_df.shape)
print('\nlabel counts (train):')
print(train_df['sentiment'].value_counts())
train_df.head()

train shape: (27480, 2)
test  shape: (3534, 2)

label counts (train):
sentiment
neutral     11117
positive     8582
negative     7781
Name: count, dtype: int64


,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative


### Downsample the training set

27k rows with a transformer on Colab's free T4 is slow. We take a balanced sample of 2k per class (6k total)

In [22]:
PER_CLASS = 5000

train_df = (
    train_df.groupby('sentiment', group_keys=False)
    .apply(lambda g: g.sample(n=min(PER_CLASS, len(g)), random_state=42))
    .reset_index(drop=True)
)
print(train_df['sentiment'].value_counts())

sentiment
negative    5000
neutral     5000
positive    5000
Name: count, dtype: int64


/tmp/ipykernel_11952/2163202360.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(PER_CLASS, len(g)), random_state=42))


## 3. Encode labels and split off a validation set

In [23]:
from sklearn.model_selection import train_test_split

LABELS = ['negative', 'neutral', 'positive']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

train_df['label'] = train_df['sentiment'].map(label2id)
test_df['label']  = test_df['sentiment'].map(label2id)

train_df = train_df.dropna(subset=['label'])
test_df  = test_df.dropna(subset=['label'])
train_df['label'] = train_df['label'].astype(int)
test_df['label']  = test_df['label'].astype(int)

train_df, val_df = train_test_split(
    train_df, test_size=0.1, random_state=42, stratify=train_df['label']
)

print('train:', train_df.shape, 'val:', val_df.shape, 'test:', test_df.shape)

train: (13500, 3) val: (1500, 3) test: (3534, 3)


## 4. Tokenize with DistilBERT's tokenizer

We wrap each split in a HuggingFace `Dataset` and map the tokenizer over it.

In [24]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(df):
    return Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))

train_ds = to_ds(train_df)
val_ds   = to_ds(val_df)
test_ds  = to_ds(test_df)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/13500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/3534 [00:00<?, ? examples/s]

## 5. Load pre-trained DistilBERT with a 3-class head

In [25]:
import torch
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Fine-tune with the HuggingFace `Trainer`

In [26]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(pred):
    preds = pred.predictions.argmax(axis=1)
    labels = pred.label_ids
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }

args = TrainingArguments(
    output_dir='distilbert-sentiment',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.574746,0.546893,0.781333,0.782961
2,0.436561,0.534493,0.787333,0.788023
3,0.364418,0.555764,0.793333,0.794638


TrainOutput(global_step=1266, training_loss=0.49950410178487337, metrics={'train_runtime': 66.9154, 'train_samples_per_second': 605.242, 'train_steps_per_second': 18.919, 'total_flos': 443108245865256.0, 'train_loss': 0.49950410178487337, 'epoch': 3.0})

## 7. Evaluate on the test set — classification report

In [27]:
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(test_ds)
preds = pred.predictions.argmax(axis=1)
labels = pred.label_ids

print('Confusion matrix (rows=true, cols=pred):')
print(confusion_matrix(labels, preds))
print('\nClassification report:')
print(classification_report(labels, preds, target_names=LABELS, digits=4))

Confusion matrix (rows=true, cols=pred):
[[ 815  163   23]
 [ 228 1031  171]
 [  32  155  916]]

Classification report:
              precision    recall  f1-score   support

    negative     0.7581    0.8142    0.7852      1001
     neutral     0.7643    0.7210    0.7420      1430
    positive     0.8252    0.8305    0.8278      1103

    accuracy                         0.7816      3534
   macro avg     0.7825    0.7885    0.7850      3534
weighted avg     0.7816    0.7816    0.7810      3534



## 8. Save the fine-tuned model

In [28]:
trainer.save_model('distilbert-sentiment-final')
tokenizer.save_pretrained('distilbert-sentiment-final')
print('saved to distilbert-sentiment-final/')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to distilbert-sentiment-final/


### Quick sanity check on custom inputs

In [29]:
from transformers import pipeline

clf = pipeline('text-classification', model='distilbert-sentiment-final', tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)
for t in [
    'I absolutely loved this movie, it was fantastic!',
    'The food was okay, nothing special.',
    'Worst experience ever, totally disappointed.',
]:
    print(t, '->', clf(t)[0])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

I absolutely loved this movie, it was fantastic! -> {'label': 'positive', 'score': 0.9875475168228149}
The food was okay, nothing special. -> {'label': 'neutral', 'score': 0.5067375302314758}
Worst experience ever, totally disappointed. -> {'label': 'negative', 'score': 0.981236457824707}
